In [1]:
!pip install -q --upgrade ipython
!pip install -q faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 624.2/624.2 kB 40.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 78.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.4/85.4 kB 7.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires ipython==7.34.0, but you have ipython 9.11.0 which is incompatible.
moviepy 1.0.3 requires decorator<5.0,>=4.0.2, but you have decorator 5.2.1 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 59.5 MB/s eta 0:00:00


In [2]:
import os
import sys
from pathlib import Path

from google.colab import drive, userdata
drive.mount('/content/drive')

sys.path.insert(0, "/content/drive/MyDrive/auto-hh")

%load_ext autoreload
%autoreload 2

os.environ['GROQ_API_KEY'] = userdata.get('GROQ_API_KEY')
os.environ['LLM_BASE_URL'] = 'https://api.groq.com/openai/v1'
os.environ['LLM_MODEL'] = 'llama-3.3-70b-versatile'
os.environ['RETRIEVAL_TOP_K'] = '50'
os.environ['FINAL_TOP_K'] = '5'

Mounted at /content/drive


In [3]:
sys.path.insert(0, '/content/drive/MyDrive/auto-hh/src')

In [4]:
from core import LetterGenerator, LLMMode
from app import App
from schemas import Resume

In [5]:
app = App(
    bi_encoder_name="BAAI/bge-m3",
    cross_encoder_model="BAAI/bge-reranker-v2-m3",
    faiss_path="/content/drive/MyDrive/auto-hh/faiss_index",
    model_path="/content/drive/MyDrive/auto-hh/models/BiEncoder",
    retrieval_top_k=50,
    final_top_k=5,
    min_score=0.0,
    llm_mode=LLMMode.API,
)

print("Application инициализирован")

📥 Загрузка модели из: /content/drive/MyDrive/auto-hh/models/BiEncoder


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/2.27G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

...

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

✅ Cross-Encoder готов!
📥 Загрузка FAISS индекса...
✅ Тексты загружены и очищены: 4499
✅ Загружено: 4,499 векторов
Application инициализирован


In [6]:
print(f"Вакансий: {app.get_stats()['total_vacancies']}")

Вакансий: 4499


In [7]:
resume = Resume(
    resume_id=123,
    job_title="Python разработчик",
    exp_text="3 года бэкенда",
    skills_res="Python, Django",
    about_me="Опыт разработки",
)

result = app.matcher.match(resume, generate_letters=True)
print(result)

MatchResult(resume_id=123, status='success', matches=[
  VacancyMatch(id=118847757, job_title='Python разработчик', score=0.9996, salary='Не указано', cover_letter=Я заинтересован возможностью присоединиться к вашей команде в качестве Python разработчика, где я смогу применить свои навыки и опыт для создания высококачественных проектов. С более чем 3-летним опытом разработки бэкенда на Python и глубоким знанием фреймворка Django, я уверен, что могу внести свой вклад в успешную реализацию ваших проектов.

В течение моей карьеры я успешно разработал и реализовал несколько проектов, включая сложные веб-приложения и сервисы. Одним из заметных достижений является создание высоконагруженного сервиса с использованием Django, который показал высокую эффективность и масштабируемость. Кроме того, я успешно оптимизировал код существующего проекта, что привело к значительному увеличению производительности.

Я с интересом отношусь к возможности работать с фреймворками FastApi и Flask, а также к раб

In [8]:
# Проверка данных из Retriever
search_text = resume.to_search_text()
raw_results = app.matcher.retriever.search(search_text)
print("📄 Что возвращает Retriever:")
print(raw_results[0].keys() if raw_results else "Пусто!")
print(raw_results[0] if raw_results else "")

📄 Что возвращает Retriever:
dict_keys(['idx', 'vacancy_id', 'score', 'text', 'target_role', 'grade', 'title', 'company'])
{'idx': 1479, 'vacancy_id': 118847757, 'score': 0.9995904564857483, 'text': 'ВАКАНСИЯ: Python разработчик. ЗАРПЛАТА: Не указано. ОПЫТ: От 3 до 6 лет. НАВЫКИ: Не указано. ОПИСАНИЕ: В Т-Банке Python является одним из самых распространенных языков разработки. На нем мы пишем многие сложные и нагруженные проекты. Поэтому ищем опытных разработчиков Python, которые смогут привнести экспертизу и вывести наши продукты на новый уровень Нам всегда есть, что предложить, начиная с внутренних проектов, заканчивая широко востребованными клиентскими продуктами Обязанности Проектирование компонентов систем Организация процесса разработки Написание и ревью кода Написание тестов на свой код Наставление менее опытных коллег Требования Отличное знание Python 3 и опыт промышленной разработки на Python от трех лет Опыт многопоточного и асинхронного программирования Опыт работы с фреймвор